# 19b Raw Video Fine-tune

Purpose: train a direct RGB-frame DNN baseline from contact-aligned batting clips. This is the pre-fusion method for the question: does simply feeding the batting video to a trainable model beat pose/object sequences, frozen encoders, VLM features, and player-season priors?

Runtime: L4 recommended. Use `tiny3d` for a small pilot, or `torchvision_r3d18` for the configured real run.

Outputs:
- `predictions/{video_finetune_run_id}/predictions_v1.parquet`
- `predictions/{video_finetune_run_id}/metrics_v1.json`
- `reports/preflight/raw_video_finetune_{video_finetune_run_id}.json`
- `models/video/{video_finetune_run_id}/checkpoint.pt`

Next: `25_method_evaluation.ipynb` for pre-fusion comparison, then optionally `15_full_fusion.ipynb`.

In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import subprocess
import sys


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return

    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    try:
        drive.mount(str(mountpoint))
    except ValueError as exc:
        message = str(exc)
        if 'Mountpoint must not already contain files' in message or 'already mounted' in message:
            print(f'Drive mount warning: {message}')
            if not (mountpoint / 'MyDrive').exists():
                drive.mount(str(mountpoint), force_remount=True)
        else:
            raise
    except Exception as exc:
        print(f'Colab Drive mount skipped or failed: {exc}')


os.environ.setdefault('BATTING_CODE_ROOT', '/content/drive/MyDrive/codex/batting_codex_handoff')


def _is_repo_dir(path: Path) -> bool:
    return (
        (path / 'src' / 'sport_pipeline' / '__init__.py').exists()
        and (path / 'configs').exists()
        and (path / 'notebooks').exists()
    )


def _resolve_repo_dir() -> Path:
    fixed = Path(os.environ['BATTING_CODE_ROOT'])
    mydrive = Path('/content/drive/MyDrive')
    candidates = [
        fixed,
        Path('/content/drive/MyDrive/codex/batting_codex_handoff'),
        Path('/content/drive/MyDrive/batting_codex_handoff'),
        Path.cwd(),
        *Path.cwd().parents,
    ]
    checked = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            return candidate
    raise FileNotFoundError(
        'sport_pipeline repo が見つかりません。Drive mount と repo 配置を確認してください.\n'
        f'BATTING_CODE_ROOT={fixed}\n'
        '推奨配置: /content/drive/MyDrive/codex/batting_codex_handoff\n'
        '別配置の場合は notebook の最初の cell より前に次を実行してください.\n'
        '  %env BATTING_CODE_ROOT=/content/drive/MyDrive/あなたの配置/batting_codex_handoff\n'
        f'MyDrive exists={mydrive.exists()}\n'
        '確認した候補:\n- ' + '\n- '.join(checked)
    )


_mount_drive_if_needed()
REPO_DIR = _resolve_repo_dir()
os.environ['BATTING_CODE_ROOT'] = str(REPO_DIR)
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ.get('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME
BASE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)
src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if importlib.util.find_spec('sport_pipeline') is None:
    raise ModuleNotFoundError(f'sport_pipeline import failed: {src_dir}')

from sport_pipeline.pipeline.run_profile import load_run_profile, resolve_statcast_date_range, run_id, stage_settings

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
START_DATE, END_DATE = resolve_statcast_date_range(RUN_PROFILE)
FULL_RUN_ID = run_id(RUN_PROFILE, 'full_run_id')
PREDICTION_RUN_ID = run_id(RUN_PROFILE, 'video_finetune_run_id', 'video_raw_finetune_mlb_2024_2026_v2')
RAW_SETTINGS = stage_settings(RUN_PROFILE, 'raw_video_finetune')

print('REPO_DIR =', REPO_DIR)
print('BASE_DIR =', BASE_DIR)
print('RUN_PROFILE_PATH =', RUN_PROFILE_PATH)
print('STATCAST_DATE_RANGE =', START_DATE, 'to', END_DATE)
print('FULL_RUN_ID =', FULL_RUN_ID)
print('PREDICTION_RUN_ID =', PREDICTION_RUN_ID)
print('RAW_SETTINGS =', RAW_SETTINGS)


In [ ]:
INSTALL_TORCHVISION = bool(RAW_SETTINGS.get('install_torchvision_default', True))
if INSTALL_TORCHVISION:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchvision', 'opencv-python-headless'])

print('torch available =', importlib.util.find_spec('torch') is not None)
print('torchvision available =', importlib.util.find_spec('torchvision') is not None)
print('cv2 available =', importlib.util.find_spec('cv2') is not None)


In [ ]:
from sport_pipeline.artifact_check import check_artifacts
from sport_pipeline.runtime.device import summarize_runtime_device

print(summarize_runtime_device(prefer_gpu=True, require_gpu=False))
required = ['manifests/bbe_events_v1.parquet', f'clips/{FULL_RUN_ID}/clips_v1.parquet']
print(check_artifacts(BASE_DIR, required))


In [ ]:
from sport_pipeline.models.video.raw_finetune import run_raw_video_finetune

outputs = run_raw_video_finetune(
    BASE_DIR,
    clip_run_id=FULL_RUN_ID,
    prediction_run_id=PREDICTION_RUN_ID,
    max_clips=RAW_SETTINGS.get('max_clips', 500),
    num_frames=int(RAW_SETTINGS.get('num_frames', 16)),
    image_size=int(RAW_SETTINGS.get('image_size', 112)),
    batch_size=int(RAW_SETTINGS.get('batch_size', 2)),
    max_epochs=int(RAW_SETTINGS.get('max_epochs', 5)),
    learning_rate=float(RAW_SETTINGS.get('learning_rate', 1e-4)),
    model_family=str(RAW_SETTINGS.get('model_family', 'tiny3d')),
    pretrained=bool(RAW_SETTINGS.get('pretrained', False)),
    freeze_backbone=bool(RAW_SETTINGS.get('freeze_backbone', False)),
    allow_model_download=bool(RAW_SETTINGS.get('allow_model_download', False)),
    hidden_dim=int(RAW_SETTINGS.get('hidden_dim', 128)),
    dropout=float(RAW_SETTINGS.get('dropout', 0.20)),
    device=str(RAW_SETTINGS.get('device', 'auto')),
    require_non_empty=bool(RAW_SETTINGS.get('require_non_empty', True)),
    resume=bool(RAW_SETTINGS.get('resume', True)),
    cache_dir=CACHE_DIR,
    cache_inputs=bool(RAW_SETTINGS.get('cache_inputs', False)),
    cache_num_workers=int(RAW_SETTINGS.get('cache_num_workers', 4)),
    cache_min_free_disk_gb=float(RAW_SETTINGS.get('cache_min_free_disk_gb', 20)),
    cache_max_file_mb=RAW_SETTINGS.get('cache_max_file_mb'),
    dataloader_num_workers=int(RAW_SETTINGS.get('dataloader_num_workers', 0)),
)
for name, path in outputs.items():
    print(name, '->', path, 'exists=', path.exists())


In [ ]:
expected = [
    f'predictions/{PREDICTION_RUN_ID}/predictions_v1.parquet',
    f'predictions/{PREDICTION_RUN_ID}/metrics_v1.json',
    f'reports/preflight/raw_video_finetune_{PREDICTION_RUN_ID}.json',
    f'reports/preflight/raw_video_finetune_{PREDICTION_RUN_ID}_progress.json',
    f'models/video/{PREDICTION_RUN_ID}/checkpoint.pt',
]
print(check_artifacts(BASE_DIR, expected))
print('NEXT: notebooks/25_method_evaluation.ipynb for visual/mechanics-only comparison.')
